In [ ]:
!pip install torch torchvision torchaudio
!pip install torch-geometric
!pip install networkx matplotlib
!pip install numpy --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 87.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving edges.csv to edges.csv
Saving nodes.csv to nodes.csv


### **Pre-Processing Dataset**

In [ ]:
import numpy as np
import pandas as pd

# Baca nodes.csv
nodes = pd.read_csv('nodes.csv')
# Tampilkan tabel nodes
print("=== Nodes CSV ===")
display(nodes)

=== Nodes CSV ===


,node_id,main_label,node_name
0,2,Gejala,Pola_serangannya_merata
1,3,Gejala,Adanya_telur
2,6,Gejala,Adanya_embun_jelaga
3,8,Gejala,Daun_berkerut
4,21,Penyakit,Begomovirus
5,23,Penyakit,Chilli_Veinal_Mottle_Virus
6,34,Gejala,Tanaman_kerdil
7,35,Gejala,Layunya_perlahan
8,43,Gejala,Daun_melengkung
9,53,Gejala,Adanya_larva


In [ ]:
# Baca edges.csv
edges = pd.read_csv('edges.csv')
# Tampilkan tabel edges
print("=== Edges CSV ===")
display(edges)

=== Edges CSV ===


,source,target,relation
0,21,34,menyebabkanGejala
1,21,78,menyebabkanGejala
2,21,171,menyebabkanGejala
3,21,280,menyebabkanGejala
4,23,2,menyebabkanGejala
5,23,171,menyebabkanGejala
6,23,248,menyebabkanGejala
7,154,2,menyebabkanGejala
8,154,35,menyebabkanGejala
9,154,124,menyebabkanGejala


#### **Tambah node 'Unknown_Penyakit' untuk mengelola gejala umum**
Tujuannya agar saat gejala yang dimasukkan tidak mengandung spesifikasi/gejala utama yg dimiliki suatu penyakit (inputan hanya berisi gejala umum), maka sistem tidak memaksa utk memilih penyakit, sehingga mendorong untuk pemeriksaan lanjut.

In [ ]:
nodes = pd.read_csv('nodes.csv')
edges = pd.read_csv('edges.csv')

# Cari relasi penyakit - gejala
edges_gejala = edges[edges['relation'].str.contains('menyebabkanGejala', case=False, na=False)]

# Hitung berapa kali gejala muncul
gejala_counts = edges_gejala['target'].value_counts()

# Ambil gejala yang muncul pada lebih dr sama dgn 2 penyakit
threshold_umum = 2
gejala_umum = gejala_counts[gejala_counts >= threshold_umum].index.tolist()

print(f"Total gejala umum: {len(gejala_umum)}")
print("Daftar gejala umum:", gejala_umum)

Total gejala umum: 6
Daftar gejala umum: [171, 2, 280, 35, 124, 200]


In [ ]:
# Cek
if "Gejala_Umum" not in nodes['node_name'].values:
    new_nodes = pd.DataFrame([
        {"node_id": nodes['node_id'].max() + 1, "node_name": "Gejala_Umum", "main_label": "Konsep"},
        {"node_id": nodes['node_id'].max() + 2, "node_name": "Unknown_Penyakit", "main_label": "Penyakit"}
    ])
    nodes = pd.concat([nodes, new_nodes], ignore_index=True)

In [ ]:
# Ambil ID node yang baru ditambahkan
id_gejala_umum = nodes.loc[nodes['node_name']=="Gejala_Umum", 'node_id'].values[0]
id_unknown_penyakit = nodes.loc[nodes['node_name']=="Unknown_Penyakit", 'node_id'].values[0]

# Buat edge baru
edges_gejala_umum = pd.DataFrame([
    {"source": g, "target": id_gejala_umum, "relation": "termasukDalam"} for g in gejala_umum
])

edge_unknown = pd.DataFrame([{
    "source": id_unknown_penyakit,
    "target": id_gejala_umum,
    "relation": "menyebabkanGejala"
}])

edges_new = pd.concat([edges, edges_gejala_umum, edge_unknown], ignore_index=True)

In [ ]:
nodes.to_csv("nodes_updated.csv", index=False)
edges_new.to_csv("edges_updated.csv", index=False)

print("done.")

done.


### **Embedding**

In [ ]:
!pip uninstall gensim -y
!pip install gensim
!pip uninstall node2vec -y
!pip install node2vec

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 15.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.4
    Uninstalling numpy-2.3.4:
      Successfully uninstalled numpy-2.3.4
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.2
    Uninstalling scipy-1.16.2:
      Successfully uninstalled scipy-1.16.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tsfre

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
from node2vec import Node2Vec

# Load CSV
nodes = pd.read_csv('nodes_updated.csv')
edges = pd.read_csv('edges_updated.csv')

# Filter node
nodes_in_edges = set(edges['source']).union(set(edges['target']))
nodes_filtered = nodes[nodes['node_id'].isin(nodes_in_edges)].reset_index(drop=True)

print(f"Jumlah node awal: {len(nodes)}")
print(f"Jumlah node yang relevan (ada di edges): {len(nodes_filtered)}")

# Buat NetworkX graph
G = nx.from_pandas_edgelist(edges, source='source', target='target', create_using=nx.DiGraph())
G.add_nodes_from(nodes_filtered['node_id'].tolist())

Jumlah node awal: 40
Jumlah node yang relevan (ada di edges): 23


In [ ]:
# Buat tabel penyakit gejala
penyakit_nodes = nodes_filtered[nodes_filtered['main_label'] == 'Penyakit']
gejala_nodes = nodes_filtered[nodes_filtered['main_label'] == 'Gejala']

table = []
for p_id, p_name in zip(penyakit_nodes['node_id'], penyakit_nodes['node_name']):
    connected_gejala_ids = list(G.successors(p_id))  # edge outgoing dari penyakit
    connected_gejala_names = nodes_filtered[nodes_filtered['node_id'].isin(connected_gejala_ids)]['node_name'].tolist()

    table.append({
        'Penyakit': p_name,
        'Gejala terkait': ', '.join(connected_gejala_names) if connected_gejala_names else '-'
    })

from IPython.display import display
table_df = pd.DataFrame(table)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)

print("Tabel Penyakit → Gejala terkait:\n")
display(table_df)

Tabel Penyakit → Gejala terkait:



,Penyakit,Gejala terkait
0,Begomovirus,"Tanaman_kerdil, Daun_mengeriting, Mosaik_pada_daun, Pola_serangannya_spot"
1,Chilli_Veinal_Mottle_Virus,"Pola_serangannya_merata, Mosaik_pada_daun, Tulang_daun_menguning"
2,Layu_Fusarium,"Pola_serangannya_merata, Layunya_perlahan, Daun_warna_kekuningan, Tanaman_layu"
3,Layu_Bakteri,"Pola_serangannya_merata, Layunya_perlahan, Daun_warna_kekuningan, Tanaman_layu, Batang_busuk_berbau"
4,Antraknosa,"Busuk_bentuk_lingkaran, Busuk_pada_buah_matang, Lingkaran_busuk_hitam, Lingkaran_busuk_cincin"
5,Virus_Mosaik_Alfalfa,"Mosaik_pada_daun, Mosaik_pada_buah, Pola_serangannya_spot"
6,Unknown_Penyakit,Gejala_Umum


In [ ]:
# Lihat node yg terkait gejala umum
konsep_nodes = nodes_filtered[nodes_filtered['main_label'] == 'Konsep']

table_konsep = []
for k_id, k_name in zip(konsep_nodes['node_id'], konsep_nodes['node_name']):
    connected_ids = list(G.successors(k_id)) + list(G.predecessors(k_id))
    connected_info = nodes_filtered[nodes_filtered['node_id'].isin(connected_ids)][['node_name', 'main_label']]

    detail_list = [f"{row['node_name']} ({row['main_label']})" for _, row in connected_info.iterrows()]

    table_konsep.append({
        'Konsep': k_name,
        'Jumlah node terhubung': len(connected_info),
        'Node terkait': ', '.join(detail_list) if len(detail_list) > 0 else '-'
    })

konsep_df = pd.DataFrame(table_konsep)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)

print("Tabel Konsep → Node terkait:\n")
display(konsep_df)

Tabel Konsep → Node terkait:



,Konsep,Jumlah node terhubung,Node terkait
0,Gejala_Umum,7,"Pola_serangannya_merata (Gejala), Layunya_perlahan (Gejala), Daun_warna_kekuningan (Gejala), Mosaik_pada_daun (Gejala), Tanaman_layu (Gejala), Pola_serangannya_spot (Gejala), Unknown_Penyakit (Penyakit)"


In [ ]:
node2vec = Node2Vec(G, dimensions=64, walk_length=10, num_walks=100, workers=2, seed=42)

# Train
model = node2vec.fit(window=5, min_count=1, batch_words=4)

node_embeddings = {str(node): model.wv[str(node)] for node in G.nodes()}

# Node feature matrix X
node_list = nodes_filtered['node_id'].tolist()
X = np.array([node_embeddings[str(n)] for n in node_list])
print("Shape node feature matrix X:", X.shape)

# Adjacency matrix A
A = nx.to_numpy_array(G, nodelist=node_list)
print("Shape adjacency matrix A:", A.shape)

# Cek
print("Node feature matrix X (5 node pertama):\n", X[:5])
print("Adjacency matrix A (5x5 top-left):\n", A[:5, :5])

Computing transition probabilities:   0%|          | 0/23 [00:00<?, ?it/s]

Shape node feature matrix X: (23, 64)
Shape adjacency matrix A: (23, 23)
Node feature matrix X (5 node pertama):
 [[-2.84582423e-03  1.05017629e-02  1.62647367e-02 -6.38737390e-03
  -1.41613593e-03 -9.48584918e-03  6.31788280e-03  5.09796618e-03
   1.06510175e-02  9.39680357e-03  1.55779608e-02  1.47192087e-02
   1.26321437e-02 -1.04954066e-02 -1.48937730e-02 -5.67281037e-04
  -4.79707075e-03  1.26941102e-02  8.98392871e-03 -2.44321139e-03
   2.74478714e-03  2.84057553e-03  1.29491063e-02 -1.50299445e-02
  -8.26667310e-05  5.51520148e-03 -1.92158169e-03  1.30979894e-02
   1.34202903e-02  9.86266602e-03 -1.12604036e-03  1.21258246e-02
  -1.34442290e-02  4.59324149e-03 -6.85709529e-03 -8.02577473e-03
   5.87482704e-03  8.11259355e-03  1.26087591e-02 -9.07265395e-03
   1.15180407e-02  1.08922720e-02 -5.96467592e-03 -1.38748661e-02
   8.38693418e-03  9.84106027e-03 -1.32324500e-03 -1.01556098e-02
  -1.09802252e-02 -3.39010055e-03  7.95252714e-03 -5.37382066e-03
  -1.47941839e-02  6.0508064

In [ ]:
nodes_filtered.to_csv('nodes_filtered.csv', index=False)
edges_filtered = edges[edges['source'].isin(nodes_filtered['node_id']) & edges['target'].isin(nodes_filtered['node_id'])]
edges_filtered.to_csv('edges_filtered.csv', index=False)

In [ ]:
np.save('X.npy', X)
np.save('A.npy', A)

# **(B) GCN-DIAGNOSE**

### **GRAPH NEURAL NETWORK (GCN)**
##### **Dengan model simpleGCN**

In [ ]:
disease_nodes = nodes_filtered[nodes_filtered['main_label']=='Penyakit']
disease_ids = disease_nodes['node_id'].tolist()
disease_names = disease_nodes['node_name'].tolist()

node_id_to_idx = {nid: idx for idx, nid in enumerate(nodes_filtered['node_id'].tolist())}

In [ ]:
import torch
from torch_geometric.nn import GCNConv

X_tensor = torch.tensor(X, dtype=torch.float)
A_tensor = torch.tensor(A, dtype=torch.float)

class SimpleGCN(torch.nn.Module):
    def __init__(self, in_feats, hidden_feats, out_feats):
        super().__init__()
        self.gc1 = GCNConv(in_feats, hidden_feats)
        self.gc2 = GCNConv(hidden_feats, out_feats)

    def forward(self, x, edge_index):
        x = self.gc1(x, edge_index).relu()
        x = self.gc2(x, edge_index)
        return x

# Convert adjacency matrix to edge_index format PyG
edge_index = torch.tensor(np.array(np.nonzero(A)), dtype=torch.long)

model = SimpleGCN(in_feats=X.shape[1], hidden_feats=32, out_feats=32)
GCN_embeddings = model(X_tensor, edge_index).detach().numpy()

#### **REASONING / DIAGNOSA**

##### ***menggunakan attention**

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def diagnose_embeddings_attention(gejala_input, nodes_filtered, GCN_embeddings, top_k=1, threshold_new=30):
    node_name_to_idx = {name: idx for idx, name in enumerate(nodes_filtered['node_name'].tolist())}

    # Ambil embedding gejala input
    gejala_idx = [node_name_to_idx[g] for g in gejala_input if g in node_name_to_idx]
    if len(gejala_idx) == 0:
        raise ValueError("Tidak ada gejala input yang valid")

    gejala_embs = GCN_embeddings[gejala_idx]

    # === Perbedaan utama di sini ===
    # Hitung attention weight tiap gejala
    # Misalnya pakai vektor bobot w yang dilatih atau random kecil (karena ini inference)
    w = np.random.randn(gejala_embs.shape[1])
    scores = np.exp(np.dot(gejala_embs, w))
    weights = scores / np.sum(scores)

    # Ambil kombinasi embedding gejala berbobot attention
    gejala_emb = np.sum(weights[:, np.newaxis] * gejala_embs, axis=0)

    # Ambil semua node penyakit
    disease_nodes = nodes_filtered[nodes_filtered['main_label'] == 'Penyakit']
    disease_idx = [node_name_to_idx[name] for name in disease_nodes['node_name'].tolist()]
    disease_names = disease_nodes['node_name'].tolist()

    disease_emb = GCN_embeddings[disease_idx]
    sim = cosine_similarity(gejala_emb.reshape(1, -1), disease_emb).flatten() * 100
    sim_dict = dict(zip(disease_names, sim))

    sorted_disease = sorted(sim_dict.items(), key=lambda x: x[1], reverse=True)
    main_disease, main_score = sorted_disease[0]
    new_flag = main_score < threshold_new

    print(f"Hasil diagnosis: {main_disease} ({main_score:.1f}%)")
    if new_flag:
        print(f"Terdapat indikasi penyakit baru (keyakinan < {threshold_new}%)")

    return main_disease, main_score, new_flag, sorted_disease


In [ ]:
# Contoh penggunaan
gejala_input_user = ['Mosaik_pada_daun', 'Pola_serangannya_spot', 'Mosaik_pada_buah']
main_disease, main_score, new_flag, sorted_disease = diagnose_embeddings_attention(
    gejala_input=gejala_input_user,
    nodes_filtered=nodes_filtered,
    GCN_embeddings=GCN_embeddings,  # hasil dari model GCN sebelumnya
    top_k=3,                        # menampilkan 3 penyakit teratas
    threshold_new=30                # ambang keyakinan utk indikasi penyakit baru
)

# --- Tampilkan hasilnya ---
print("\n=== Hasil Diagnosis Sistem Pakar ===")
print(f"Penyakit utama terdeteksi: {main_disease}")
print(f"Tingkat keyakinan: {main_score:.2f}%")

if new_flag:
    print("Indikasi penyakit baru (keyakinan rendah)")
else:
    print("Diagnosis meyakinkan berdasarkan gejala yang dipilih.")

# print("\nPeringkat penyakit serupa:")
# for i, (dis, score) in enumerate(sorted_disease[:3], 1):
#     print(f"{i}. {dis} — {score:.2f}%")

Hasil diagnosis: Virus_Mosaik_Alfalfa (77.0%)

=== Hasil Diagnosis Sistem Pakar ===
Penyakit utama terdeteksi: Virus_Mosaik_Alfalfa
Tingkat keyakinan: 76.96%
Diagnosis meyakinkan berdasarkan gejala yang dipilih.


##### ***menggunakan attention (rev)**

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

def attention_pooling(embeddings):
    if embeddings.shape[0] == 1:
        return embeddings[0]

    mean_emb = embeddings.mean(axis=0, keepdims=True)
    scores = np.dot(embeddings, mean_emb.T)
    attn_weights = F.softmax(torch.tensor(scores), dim=0).numpy()

    weighted_emb = np.sum(attn_weights * embeddings, axis=0)
    return weighted_emb

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def diagnose_embeddings_attention_rev(gejala_input, nodes_filtered, edges, GCN_embeddings, top_k=1, threshold_new=75):
    node_name_to_idx = {name: idx for idx, name in enumerate(nodes_filtered['node_name'].tolist())}

    gejala_idx = [node_name_to_idx[g] for g in gejala_input if g in node_name_to_idx]
    if len(gejala_idx) == 0:
        raise ValueError("Tidak ada gejala input yang valid!")

    gejala_emb = attention_pooling(GCN_embeddings[gejala_idx])

    disease_nodes = nodes_filtered[nodes_filtered['main_label'] == 'Penyakit']
    disease_idx = [node_name_to_idx[name] for name in disease_nodes['node_name'].tolist()]
    disease_names = disease_nodes['node_name'].tolist()

    disease_emb = GCN_embeddings[disease_idx]

    # Hitung cosine similarity input gejala tiap penyakit
    sim = cosine_similarity(gejala_emb.reshape(1, -1), disease_emb).flatten() * 100
    sim_dict = dict(zip(disease_names, sim))

    # Set hasilnya
    sorted_disease = sorted(sim_dict.items(), key=lambda x: x[1], reverse=True)
    main_disease, main_score = sorted_disease[0]

    if main_score < threshold_new:
        main_disease = "Unknown_Penyakit"
        print(f"Hasil diagnosis: {main_disease}")
    else:
        print(f"Hasil diagnosis: {main_disease}")

    return main_disease, main_score, sorted_disease

In [ ]:
gejala_input = ['Batang_busuk_berbau']

main_disease, main_score, sorted_disease = diagnose_embeddings_attention_rev(
    gejala_input=gejala_input,
    nodes_filtered=nodes_filtered,
    edges=edges,
    GCN_embeddings=GCN_embeddings,
    top_k=3,
    threshold_new=70
)

print("\n=== Hasil Diagnosis Sistem Pakar ===")
print(f"Penyakit utama terdeteksi: {main_disease}")
if main_disease == "Unknown_Penyakit":
    print("Indikasi penyakit baru (keyakinan rendah)")
    print("\nPeringkat penyakit dengan keyakinan sangat rendah:")
    for i, (dis, score) in enumerate(sorted_disease[:2], 1):
        print(f"{i}. {dis} — {score:.2f}%")
else:
    print(f"Tingkat keyakinan: {main_score:.2f}%")

Hasil diagnosis: Layu_Bakteri

=== Hasil Diagnosis Sistem Pakar ===
Penyakit utama terdeteksi: Layu_Bakteri
Tingkat keyakinan: 99.11%
